# Li-ion Battery Aging — EDA with a Thermal Engineer's Eye 🔋🔥

Hi! I'm a CFD engineer specializing in **battery thermal management and combustion**, now applying ML to battery degradation.

In this notebook we explore the classic **NASA PCoE Li-ion battery aging dataset** (cells cycled to failure at controlled temperatures) and ask:

1. How does **capacity fade** progress over the life of a cell?
2. How do **discharge voltage curves** change as the cell ages?
3. What does the **cell temperature** tell us about rising internal resistance?

This is Part 1 of a series. Part 2 will build features from these cycles and predict **State of Health (SOH) / Remaining Useful Life (RUL)**.

*If you work on batteries or BMS and see something I should dig into — leave a comment!*

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
sns.set_palette('viridis')

# Explore what the attached dataset contains — path depends on which
# NASA battery dataset you attached. List everything first:
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files[:10]:
        print(os.path.join(root, f))
    if len(files) > 10:
        print(f'... and {len(files)-10} more in {root}')

## 1. Load the data

Two common versions of this dataset exist on Kaggle:

- **CSV version** (e.g. `patrickfleith/nasa-battery-dataset`): a `metadata.csv` indexing thousands of per-cycle CSV files.
- **Raw `.mat` version**: original MATLAB files (`B0005.mat`, `B0006.mat`, ...).

Use the loader matching your attached dataset (delete the other).

In [ ]:
# ---------- OPTION A: CSV version ----------
DATA_DIR = '/kaggle/input/nasa-battery-dataset/cleaned_dataset'  # adjust after listing files above

meta = pd.read_csv(os.path.join(DATA_DIR, 'metadata.csv'))
print(meta.shape)
meta.head()

In [ ]:
# Keep the 4 classic cells and discharge cycles only
CELLS = ['B0005', 'B0006', 'B0007', 'B0018']

dis = meta[(meta['battery_id'].isin(CELLS)) & (meta['type'] == 'discharge')].copy()
dis = dis.sort_values(['battery_id', 'test_id']).reset_index(drop=True)

# Cycle number per cell
dis['cycle'] = dis.groupby('battery_id').cumcount() + 1

# Capacity column: present directly in metadata for discharge rows
dis['Capacity'] = pd.to_numeric(dis['Capacity'], errors='coerce')
dis = dis.dropna(subset=['Capacity'])
print(dis.groupby('battery_id')['cycle'].max())

In [ ]:
# Helper to load one cycle's time-series (voltage, current, temperature)
def load_cycle(filename):
    df = pd.read_csv(os.path.join(DATA_DIR, 'data', filename))
    # Typical columns: Voltage_measured, Current_measured,
    # Temperature_measured, Time
    return df

example = load_cycle(dis.iloc[0]['filename'])
example.head()

In [ ]:
# ---------- OPTION B: raw .mat version ----------
# from scipy.io import loadmat
#
# def load_mat_cell(path, cell_name):
#     mat = loadmat(path, simplify_cells=True)
#     cycles = mat[cell_name]['cycle']
#     rows, series = [], []
#     cyc_no = 0
#     for c in cycles:
#         if c['type'] != 'discharge':
#             continue
#         cyc_no += 1
#         d = c['data']
#         rows.append({'cycle': cyc_no,
#                      'Capacity': float(d['Capacity']),
#                      'ambient_T': c['ambient_temperature']})
#         series.append(pd.DataFrame({
#             'cycle': cyc_no,
#             'time': d['Time'],
#             'V': d['Voltage_measured'],
#             'I': d['Current_measured'],
#             'T': d['Temperature_measured']}))
#     return pd.DataFrame(rows), pd.concat(series, ignore_index=True)
#
# cap5, ts5 = load_mat_cell('/kaggle/input/.../B0005.mat', 'B0005')

## 2. Capacity fade — the aging signature

Capacity fade is driven mainly by **SEI layer growth** (consumes cyclable lithium) and **loss of active material**. Both are thermally activated processes — Arrhenius-type kinetics — which is why temperature management matters so much for battery life.

In [ ]:
fig, ax = plt.subplots()
for cell, g in dis.groupby('battery_id'):
    ax.plot(g['cycle'], g['Capacity'], label=cell, lw=2)

# End-of-life threshold: 70-80% of nominal 2 Ah is typical
ax.axhline(0.7 * 2.0, color='red', ls='--', label='EOL (70% of 2 Ah)')
ax.set_xlabel('Cycle number')
ax.set_ylabel('Discharge capacity [Ah]')
ax.set_title('Capacity fade — NASA PCoE cells')
ax.legend()
plt.show()

**Things to point out in your narrative:**
- Initial gradual fade, then a *knee point* where degradation accelerates — this knee is the holy grail of RUL prediction.
- Capacity regeneration spikes after rest periods (visible as small upward jumps) — a thermal/relaxation effect worth explaining.
- Cell-to-cell variability even under identical test conditions.

## 3. Discharge voltage curves — early vs late life

In [ ]:
cell = 'B0005'
g = dis[dis['battery_id'] == cell]
picks = g.iloc[np.linspace(0, len(g) - 1, 6).astype(int)]

fig, ax = plt.subplots()
colors = plt.cm.viridis(np.linspace(0, 1, len(picks)))
for color, (_, row) in zip(colors, picks.iterrows()):
    ts = load_cycle(row['filename'])
    ax.plot(ts['Time'], ts['Voltage_measured'],
            color=color, label=f"cycle {row['cycle']}")

ax.set_xlabel('Time [s]')
ax.set_ylabel('Voltage [V]')
ax.set_title(f'{cell}: discharge curves shrink and sag with age')
ax.legend()
plt.show()

**Narrative:** the curve gets *shorter* (less capacity) and *sags lower* (higher internal resistance → larger IR drop). That resistance growth is also a heat-generation story, which leads us to...

## 4. Temperature — the thermal engineer's section 🔥

Heat generation in a cell ≈ **ohmic/polarization heat** $I^2R_{int}$ plus **entropic heat** $IT\,\frac{dU}{dT}$.

As the cell ages, $R_{int}$ grows → for the same discharge current, the cell runs **hotter**. Higher temperature accelerates SEI growth → faster aging. This positive feedback loop is exactly why pack-level thermal design matters.

In [ ]:
fig, ax = plt.subplots()
for color, (_, row) in zip(colors, picks.iterrows()):
    ts = load_cycle(row['filename'])
    ax.plot(ts['Time'], ts['Temperature_measured'],
            color=color, label=f"cycle {row['cycle']}")

ax.set_xlabel('Time [s]')
ax.set_ylabel('Cell temperature [°C]')
ax.set_title(f'{cell}: temperature rise during discharge, early vs late life')
ax.legend()
plt.show()

In [ ]:
# Quantify it: peak temperature rise per cycle vs cycle number
rows = []
for _, row in g.iterrows():
    ts = load_cycle(row['filename'])
    rows.append({'cycle': row['cycle'],
                 'dT_peak': ts['Temperature_measured'].max()
                            - ts['Temperature_measured'].iloc[0],
                 'Capacity': row['Capacity']})
thermal = pd.DataFrame(rows)

fig, ax1 = plt.subplots()
ax1.plot(thermal['cycle'], thermal['dT_peak'], color='tab:red')
ax1.set_xlabel('Cycle')
ax1.set_ylabel('Peak ΔT during discharge [°C]', color='tab:red')
ax2 = ax1.twinx()
ax2.plot(thermal['cycle'], thermal['Capacity'], color='tab:blue')
ax2.set_ylabel('Capacity [Ah]', color='tab:blue')
ax2.grid(False)
plt.title(f'{cell}: as capacity fades, the cell runs hotter')
plt.show()

## 5. Takeaways & what's next

1. Capacity fade shows a knee point — predicting it early is the core RUL challenge.
2. Voltage sag and temperature rise both encode **internal resistance growth** — great ML features.
3. The aging–heat feedback loop is why thermal management directly buys battery life.

**Part 2** will engineer per-cycle features (discharge time, voltage slope, ΔT) and train a model to predict SOH/RUL.

*Thanks for reading — upvote if useful, and tell me in the comments which degradation mechanism you'd like explored next!*